<a href="https://colab.research.google.com/github/s24bsit044-crypto/ED05_Balochistan_Textbook_Board_Tutor_English_page_level/blob/main/ED05_Textbook_RAG_Tutor_English_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ED-05 · Textbook-Board Retrieval-Augmented Tutor with Page-Level Citation
**Balochistan Textbook Board (Grades 9–12) · AIDIRL, University of Gwadar**

This notebook builds the complete pipeline: **ingestion → structural chunking → index → hybrid retrieval → fine-tuned dense retriever → cited reader (abstain when weak) → evaluation → interface**.

### Choice of model (Model choice)
| Layer | Model | Why |
|---|---|---|
| Floor | **BM25** (+ extractive answer) | Cheap, hallucination-free, strong baseline |
| Dense retriever | **`intfloat/multilingual-e5-base`** (278M) — *fine-tuned* on our data | Urdu/English/cross-lingual, can be fine-tuned on a T4 GPU. (Larger option: `BAAI/bge-m3`) |
| Fusion | **Reciprocal Rank Fusion (BM25 + dense)** | Hybrid, robust to OCR noise |
| Reader | **`Qwen/Qwen2.5-3B-Instruct`** (restricted to retrieved passages) | Instruction-tuned, multilingual, runs on a T4 |

### An important honest note (about the data)
`dataset_statistics.json` states clearly: *"retrieval corpus, not model-training labels"*. That is, this data **contains no question-answer pairs**.
Therefore we do not "train" the reader (it is an instruction-tuned LLM that stays constrained to the passages). **We fine-tune the retriever in a self-supervised way**: an LLM generates one question from each passage → (question, passage) pair.
**The split is at book level** (train: 9 books, test: Physics 9 and Physics 11) — no book appears in both.
To check the real "correct answers" we need the teacher-gold set (Cell 14) — numbers from synthetic questions are a *proxy*, not a final claim.

**Runtime:** Colab → Runtime → Change runtime type → **T4 GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [5]:
# ===== Cell 1: Setup & data location (PC upload or Google Drive) =====
!pip -q install sentence-transformers rank_bm25 transformers accelerate pandas tqdm gradio pymupdf bitsandbytes

import os, json, re, random, pickle, time, math, collections
import numpy as np, pandas as pd
from tqdm.auto import tqdm

DATA_SOURCE = 'drive'     # Changed from 'upload' to 'drive' to connect your project to Google Drive
try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and DATA_SOURCE == 'drive':
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/ED05_Tutor'
elif IN_COLAB:
    BASE = '/content/ED05_Tutor'          # Colab temporary disk: cleared when the session ends
else:                                     # local run
    BASE = os.environ.get('ED05_BASE', './ED05_Tutor')

DATA = f'{BASE}/data'        # put passages.jsonl, train.jsonl, test.jsonl, dataset_statistics.json here
PDFS = f'{BASE}/pdfs'        # (optional) PDFs of the original books — to show page images
OUT  = f'{BASE}/outputs'     # index, fine-tuned model, results
GOLD = f'{BASE}/gold'        # teacher gold CSV
for d in (DATA, PDFS, OUT, GOLD): os.makedirs(d, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
except ImportError:
    DEVICE = 'cpu'
print('BASE =', BASE, '| device =', DEVICE)

# ---- Configuration ----
VAL_BOOK = 'btbb_12_chemistry'     # hold-out from the train books: to calibrate the threshold
N_TRAIN_QUERIES = 2500             # synthetic queries for fine-tuning (~25-40 minutes on a T4)
N_EVAL_QUERIES  = 400              # val/test per split
DENSE_MODEL  = 'intfloat/multilingual-e5-base'
READER_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
QGEN_MODEL   = 'Qwen/Qwen2.5-1.5B-Instruct'

MessageError: Error: credential propagation was unsuccessful

## 1b. Add your own dataset from your PC
Run the next cell and use the **Choose files** button to select files from your computer. You can upload any mix of these (several at once, or a single `.zip` containing them):

| What you upload | Where it goes | Notes |
|---|---|---|
| `passages.jsonl`, `train.jsonl`, `test.jsonl`, `dataset_statistics.json` | `data/` | Ready-made dataset in the same format as the original (see the required fields below) |
| `*.pdf` textbooks | `pdfs/` | **Easiest way to use your own books.** If no `passages.jsonl` exists, the notebook builds the whole dataset from the PDFs (one passage per page, with the page number kept) and splits it by book |
| `gold_eval.csv` | `gold/` | Teacher-gold questions (used in Cell 14) |

**Required fields per passage** (`jsonl`, one JSON object per line): `id`, `book_id`, `page_number`, `text`, `subject`, `class_level`, `source_file`. Optional (filled in automatically if missing): `book_title`, `chapter`.

**Needs at least 3 books** (≥2 for training — one of them is held out for validation — and ≥1 for testing). The split is by book, never by page, so no book is in both train and test.

Tip: instead of the upload button you can also drag files into the Colab **Files** panel (left sidebar) into `/content/ED05_Tutor/data` (or `pdfs`, `gold`) and then just run the cell. Re-run it any time you want to add more files.

In [ ]:
# ===== Cell 1b: Upload your own data from your PC (and build the dataset if needed) =====
import glob, shutil, zipfile, math

# ---- optional settings for your own books ----
TEST_BOOKS = []        # book_ids (or parts of the file name) to use as the TEST set, e.g. ['physics_9']. Empty = last ~20% of books (alphabetical)
BOOK_INFO  = {}        # optional manual labels, e.g. {'my_bio_book': {'subject':'Biology','class_level':'10','book_title':'Biology 10'}}

def route_file(path, name=None):
    name = name or os.path.basename(path); low = name.lower()
    if low.endswith('.zip'):
        with zipfile.ZipFile(path) as z:
            for m in z.namelist():
                if m.endswith('/') or m.startswith('__MACOSX') or os.path.basename(m).startswith('.'): continue
                z.extract(m, '/tmp/_unz'); route_file(os.path.join('/tmp/_unz', m), os.path.basename(m))
        return
    if low.endswith('.pdf'):                dest = PDFS
    elif low.startswith('gold') and low.endswith('.csv'): dest = GOLD
    elif low.endswith(('.jsonl', '.json')): dest = DATA
    else: print('skipped (unknown type):', name); return
    shutil.copy(path, os.path.join(dest, name)); print('saved ->', os.path.join(dest, name))

if IN_COLAB and DATA_SOURCE == 'upload':
    print('Choose the files from your PC (dataset files, PDFs, gold CSV, or a .zip):')
    up = files.upload()                       # opens the file picker
    for name, content in up.items():
        tmp = f'/tmp/{name}'; open(tmp, 'wb').write(content); route_file(tmp, name)

def _guess_meta(stem):
    low = stem.lower(); info = {}
    for key, subj in [('bio','Biology'),('chem','Chemistry'),('phys','Physics'),('math','Mathematics'),
                      ('eng','English'),('urdu','Urdu'),('comp','Computer Science'),('islam','Islamiat'),('pak','Pakistan Studies')]:
        if key in low: info['subject'] = subj; break
    m = re.search(r'(?<!\d)(1[0-2]|9)(?!\d)', stem); info['class_level'] = m.group(1) if m else 'unknown'
    info.setdefault('subject', 'General'); info['book_title'] = re.sub(r'[_\-]+', ' ', stem).strip().title()
    return info

def build_from_pdfs():
    import fitz
    pdfs = sorted(glob.glob(f'{PDFS}/*.pdf')); assert pdfs, f'No PDFs found in {PDFS}'
    rows, bstats = [], []
    for pdf in pdfs:
        stem = os.path.splitext(os.path.basename(pdf))[0]; bid = re.sub(r'[^a-z0-9]+', '_', stem.lower()).strip('_')
        info = {**_guess_meta(stem), **BOOK_INFO.get(bid, {})}; doc = fitz.open(pdf); with_text = 0
        for i, page in enumerate(doc):
            t = re.sub(r'[ \t]+', ' ', page.get_text()).strip()
            if len(t.split()) < 5: continue
            with_text += 1
            rows.append(dict(id=f'{bid}_p{i+1:04d}', book_id=bid, page_number=i+1, text=t, chapter='Unlabeled section',
                             source_file=os.path.basename(pdf), **info))
        bstats.append(dict(book_id=bid, page_count=len(doc), pages_with_text=with_text)); print(f'{bid}: {with_text}/{len(doc)} pages with text')
    return rows, bstats

def _split_by_book(rows):
    ids = sorted({r['book_id'] for r in rows})
    assert len(ids) >= 3, f'Need at least 3 books (found {len(ids)}: {ids}). Upload more books.'
    test = [b for b in ids if any(t.lower() in b for t in TEST_BOOKS)] if TEST_BOOKS else ids[-max(1, round(0.2*len(ids))):]
    assert test and len(ids) - len(test) >= 2, 'TEST_BOOKS leaves fewer than 2 train books'
    print('train books:', [b for b in ids if b not in test]); print('test books :', test)
    return [r for r in rows if r['book_id'] not in test], [r for r in rows if r['book_id'] in test]

def _write_jsonl(path, rows):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + '\n')

def ensure_dataset():
    pj, trj, tej, stj = (f'{DATA}/{n}' for n in ('passages.jsonl', 'train.jsonl', 'test.jsonl', 'dataset_statistics.json'))
    if not os.path.exists(pj):
        print('passages.jsonl not found -> building the dataset from your PDFs ...')
        rows, bstats = build_from_pdfs(); _write_jsonl(pj, rows)
        json.dump(dict(note='retrieval corpus built from user PDFs, not model-training labels', books=bstats), open(stj, 'w'), indent=1)
    rows = read_jsonl_(pj)
    for r in rows:                                   # fill optional fields
        r.setdefault('chapter', 'Unlabeled section'); r.setdefault('book_title', r['book_id'].replace('_', ' ').title())
    _write_jsonl(pj, rows)
    if not (os.path.exists(trj) and os.path.exists(tej)):
        print('train.jsonl / test.jsonl not found -> splitting by book ...'); tr, te = _split_by_book(rows)
        _write_jsonl(trj, tr); _write_jsonl(tej, te)
    if not os.path.exists(stj):
        bs = collections.defaultdict(set)
        for r in rows: bs[r['book_id']].add(r['page_number'])
        json.dump(dict(note='retrieval corpus', books=[dict(book_id=b, page_count=max(v), pages_with_text=len(v)) for b, v in bs.items()]), open(stj, 'w'), indent=1)
    print('Dataset ready in', DATA, '|', len(rows), 'passages')

def read_jsonl_(p): return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]
ensure_dataset()
print('PDFs   :', [os.path.basename(x) for x in glob.glob(f'{PDFS}/*.pdf')])
print('Gold   :', [os.path.basename(x) for x in glob.glob(f'{GOLD}/*.csv')])

Choose the files from your PC (dataset files, PDFs, gold CSV, or a .zip):


In [ ]:
# Install Tesseract OCR engine and wrapper libraries for processing scanned images
print("Installing OCR dependencies (this may take a minute)...")
!apt-get update -qq > /dev/null
!apt-get install -y tesseract-ocr libtesseract-dev > /dev/null
!pip -q install pytesseract pdf2image

import os, glob, shutil, re, json, collections
import fitz
import pytesseract
from PIL import Image
import io

def build_from_pdfs_with_ocr():
    pdfs = sorted(glob.glob(f'{PDFS}/*.pdf'))
    assert pdfs, f'No PDFs found in {PDFS}'
    rows, bstats = [], []

    for pdf in pdfs:
        stem = os.path.splitext(os.path.basename(pdf))[0]
        bid = re.sub(r'[^a-z0-9]+', '_', stem.lower()).strip('_')
        info = {**_guess_meta(stem), **BOOK_INFO.get(bid, {})}
        doc = fitz.open(pdf)
        with_text = 0

        print(f'Processing {bid} ({len(doc)} pages)...')
        # Process a subset (e.g., first 50 pages) to prevent T4 timeouts if scanned books are large
        pages_to_process = min(len(doc), 50)

        for i in range(pages_to_process):
            page = doc[i]
            t = re.sub(r'[ \t]+', ' ', page.get_text()).strip()

            # Fallback to OCR if page has no selectable text
            if len(t.split()) < 5:
                try:
                    pix = page.get_pixmap(dpi=100)
                    img_data = pix.tobytes("png")
                    img = Image.open(io.BytesIO(img_data))
                    t = pytesseract.image_to_string(img)
                    t = re.sub(r'[ 	]+', ' ', t).strip()
                except Exception as e:
                    t = ""

            if len(t.split()) < 5:
                continue

            with_text += 1
            rows.append(dict(
                id=f'{bid}_p{i+1:04d}',
                book_id=bid,
                page_number=i+1,
                text=t,
                chapter='Unlabeled section',
                source_file=os.path.basename(pdf),
                **info
            ))

        bstats.append(dict(book_id=bid, page_count=len(doc), pages_with_text=with_text))
        print(f'-> {bid}: extracted {with_text} pages successfully.')

    return rows, bstats

# Override original dataset building function with our robust OCR-fallback version
import sys
this_module = sys.modules[__name__]
this_module.build_from_pdfs = build_from_pdfs_with_ocr

# Force reset empty/corrupt dataset files
pj = f'{DATA}/passages.jsonl'
if os.path.exists(pj):
    try:
        if os.path.getsize(pj) == 0 or len([json.loads(l) for l in open(pj, encoding='utf-8') if l.strip()]) == 0:
            print("Removing empty passages.jsonl cache...")
            os.remove(pj)
    except Exception:
        os.remove(pj)

# Re-run dataset preparation once the files are uploaded
ensure_dataset()
print('Currently uploaded PDFs:', [os.path.basename(x) for x in glob.glob(f'{PDFS}/*.pdf')])

Installing OCR dependencies (this may take a minute)...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Removing empty passages.jsonl cache...
passages.jsonl not found -> building the dataset from your PDFs ...
Processing 1717_balochistan_board_11th_class_biology_text_book_pdf_taleem360_com_kkkrf (349 pages)...
-> 1717_balochistan_board_11th_class_biology_text_book_pdf_taleem360_com_kkkrf: extracted 48 pages successfully.
Processing 1718_balochistan_board_12th_class_biology_text_book_pdf_taleem360_com_alnw8 (424 pages)...
-> 1718_balochistan_board_12th_class_biology_text_book_pdf_taleem360_com_alnw8: extracted 50 pages successfully.
Processing 2861_9th_class_biology_btbb_new_text_book_2026_pdf_taleem360_com_1hggf (172 pages)...
-> 2861_9th_class_biology_btbb_new_text_book_2026_pdf_taleem360_com_1hggf: extracted 50 pages successfully.
train.jsonl /

In [6]:
# ===== Troubleshooting Drive Mount =====
# If this cell fails with 'credential propagation was unsuccessful':
# 1. Look at your browser address bar: check if a pop-up window or third-party cookies are blocked.
# 2. Click the 'pop-up blocked' icon, choose 'Always allow pop-ups and redirects from colab.research.google.com', and refresh.
# 3. If using Safari or Incognito mode, temporarily enable cross-site tracking/cookies in your browser settings.

from google.colab import drive
try:
    print("Requesting Google Drive permission access popup...")
    drive.mount('/content/drive', force_remount=True)
    print("Drive mounted successfully!")
except Exception as e:
    print(f"\n[ERROR] Mounting failed: {e}")
    print("Please grant permissions in the browser prompt or verify browser cookie settings.")

Requesting Google Drive permission access popup...

[ERROR] Mounting failed: Error: credential propagation was unsuccessful
Please grant permissions in the browser prompt or verify browser cookie settings.


## 1. Ingestion — page provenance and extraction audit
The most important rule of the specification: **never lose the page number**. The pipeline *fails loudly* without provenance.

In [ ]:
# ===== Cell 2: Load + provenance audit =====
def read_jsonl(p): return [json.loads(l) for l in open(p, encoding='utf-8')]

# Ensure dataset splits are completely written out first
ensure_dataset()

passages = read_jsonl(f'{DATA}/passages.jsonl')
train_p  = read_jsonl(f'{DATA}/train.jsonl')
test_p   = read_jsonl(f'{DATA}/test.jsonl')
stats    = json.load(open(f'{DATA}/dataset_statistics.json'))

# --- provenance: fail loudly ---
REQ = ['id','book_id','page_number','text','subject','class_level','source_file']
bad = [p['id'] for p in passages if any(k not in p or p[k] in (None,'') for k in REQ)]
assert not bad, f'Provenance missing for {len(bad)} passages, e.g. {bad[:3]}'
assert len({p['id'] for p in passages}) == len(passages), 'duplicate ids'

# --- split leakage check (book-level) ---
tr_books = {p['book_id'] for p in train_p}; te_books = {p['book_id'] for p in test_p}
assert not (tr_books & te_books), 'LEAK: the same book is in both train and test'
print('train books:', sorted(tr_books))
print('test books :', sorted(te_books))
if VAL_BOOK not in tr_books:
    VAL_BOOK = collections.Counter(p['book_id'] for p in train_p).most_common(1)[0][0]
    print('VAL_BOOK not in your train books -> using', VAL_BOOK)
assert len(tr_books) >= 2, 'Need at least 2 train books (one is held out for validation)'

# --- extraction-failure audit ---
books = {b['book_id']: b for b in stats['books']}
rows = []
for bid, b in books.items():
    pages = {p['page_number'] for p in passages if p['book_id'] == bid}
    missing = sorted(set(range(1, b['page_count']+1)) - pages)
    rows.append(dict(book=bid, pages=b['page_count'], with_text=b['pages_with_text'],
                     missing_pages=len(missing), missing_list=missing[:8]))
audit = pd.DataFrame(rows); display(audit)

# --- OCR noise and weak chunks ---
df = pd.DataFrame(passages)
df['n_words'] = df.text.str.split().str.len()
df['alpha_ratio'] = df.text.apply(lambda t: sum(c.isalpha() for c in t)/max(1,len(t)))
print('chunks <15 words:', (df.n_words<15).sum(), '| low alpha-ratio (<0.6):', (df.alpha_ratio<0.6).sum())
print('chapter "Unlabeled section":', (df.chapter=='Unlabeled section').mean().round(3), '→ we derive the chapter ourselves below')
print('NOTE: page_number = PDF page index (can differ from the printed page number) — the interface shows only the PDF page.')

Dataset ready in /content/ED05_Tutor/data | 148 passages
train books: ['1717_balochistan_board_11th_class_biology_text_book_pdf_taleem360_com_kkkrf', '1718_balochistan_board_12th_class_biology_text_book_pdf_taleem360_com_alnw8']
test books : ['2861_9th_class_biology_btbb_new_text_book_2026_pdf_taleem360_com_1hggf']
VAL_BOOK not in your train books -> using 1718_balochistan_board_12th_class_biology_text_book_pdf_taleem360_com_alnw8


,book,pages,with_text,missing_pages,missing_list
0,1717_balochistan_board_11th_class_biology_text...,349,48,301,"[38, 42, 51, 52, 53, 54, 55, 56]"
1,1718_balochistan_board_12th_class_biology_text...,424,50,374,"[51, 52, 53, 54, 55, 56, 57, 58]"
2,2861_9th_class_biology_btbb_new_text_book_2026...,172,50,122,"[51, 52, 53, 54, 55, 56, 57, 58]"


chunks <15 words: 2 | low alpha-ratio (<0.6): 0
chapter "Unlabeled section": 1.0 → we derive the chapter ourselves below
NOTE: page_number = PDF page index (can differ from the printed page number) — the interface shows only the PDF page.


In [ ]:
# Option 1: Native Interactive Data Table (No authentication required, highly recommended)
from google.colab import data_table
data_table.enable_dataframe_formatter()
display(audit)

# Option 2: Google Sheets (If you specifically need to view it in Google Sheets, run the code below)
# from google.colab import auth
# auth.authenticate_user() # Explicitly prompt for credentials first
# from google.colab import sheets
# sheet = sheets.InteractiveSheet(df=audit)

,book,pages,with_text,missing_pages,missing_list
0,1717_balochistan_board_11th_class_biology_text...,349,48,301,"[38, 42, 51, 52, 53, 54, 55, 56]"
1,1718_balochistan_board_12th_class_biology_text...,424,50,374,"[51, 52, 53, 54, 55, 56, 57, 58]"
2,2861_9th_class_biology_btbb_new_text_book_2026...,172,50,122,"[51, 52, 53, 54, 55, 56, 57, 58]"


## 2. Structural chunking vs fixed-window
In the provided `passages.jsonl`, chunks are page-level (≈177 words). We turn each book into a page-ordered stream (each word carries its page), then run two chunkers:
- **Fixed-window** (baseline): 200 words, no overlap.
- **Structural**: boundaries at `Example/Exercise/Activity/Summary`, `2.3 Heading` and `Chapter/Unit`; **Example + Solution in the same chunk**. Every chunk carries the chapter context and `page_start–page_end`.

And per the specification's "hard part": we count **split examples** separately, not just aggregate retrieval.

In [ ]:
# ===== Cell 3: Chunkers =====
SENT_END = tuple('.?!:;)')
HEAD_WORDS = {'Example','Exercise','Activity','Experiment','Summary'}
SKIP_BEFORE = {'in','see','of','to','from','and','the','at','on','by','with','chapter','unit'}

def book_stream(book_id):
    # Use .get() with a default fallback to handle freshly processed passages that do not have a chunk_id key
    ps = sorted([p for p in passages if p['book_id']==book_id], key=lambda p:(p['page_number'], p.get('chunk_id', 0)))
    words, pages = [], []
    for p in ps:
        w = p['text'].split()
        words += w; pages += [p['page_number']]*len(w)
    return words, pages

def find_boundaries(words):
    # return: sorted list of (word_index, kind, chapter_label)
    b, chapter = [], None
    n = len(words)
    for i, w in enumerate(words):
        prev = words[i-1] if i else ''
        sent_start = (i == 0) or prev.endswith(SENT_END) or prev.isdigit()
        wc = w.strip('.:,')
        if wc in HEAD_WORDS and sent_start:
            b.append((i, 'example' if wc=='Example' else 'section', chapter))
        elif wc in ('Chapter','Unit') and i+1<n and words[i+1].strip(':.').isdigit() and prev.lower() not in SKIP_BEFORE:
            label = ' '.join(words[i:i+6]).split('.')[0]
            if label != chapter:
                chapter = label; b.append((i, 'chapter', chapter))
        elif re.fullmatch(r'\d{1,2}\.\d{1,2}(\.\d{1,2})?', w) and i+1<n and words[i+1][:1].isupper() and sent_start:
            b.append((i, 'section', chapter))
    return b

def chapter_at(bounds, i):
    ch = None
    for j,_,c in bounds:
        if j > i: break
        if c: ch = c
    return ch

def structural_chunks(book_id, min_w=80, max_w=330, max_ex=520):
    words, pages = book_stream(book_id)
    bounds = find_boundaries(words)
    cut = sorted({0, len(words)} | {j for j,_,_ in bounds})
    kinds = {j:k for j,k,_ in bounds}
    segs = [(cut[k], cut[k+1], kinds.get(cut[k],'text')) for k in range(len(cut)-1)]
    chunks, cur = [], None
    def flush():
        nonlocal cur
        if cur: chunks.append(cur); cur=None
    for s,e,k in segs:
        L = e-s
        if k == 'example':                       # example = atomic: previous part closed, example becomes its own chunk
            flush()
            cur = dict(s=s,e=e,kind='example')
            if L <= max_ex: flush(); continue
        if cur is None: cur = dict(s=s,e=e,kind='text')
        elif (cur['e']-cur['s']) >= min_w and (cur['e']-cur['s'])+L > max_w: flush(); cur = dict(s=s,e=e,kind='text')
        else: cur['e'] = e
    flush()
    # split oversize (non-example) chunks at a sentence-ish boundary
    out = []
    for c in chunks:
        L = c['e']-c['s']; lim = max_ex if c['kind']=='example' else max_w*1.3
        if L <= lim: out.append(c); continue
        step = max_w; a = c['s']
        while a < c['e']:
            z = min(a+step, c['e'])
            while z < c['e'] and z-a < step+40 and not words[z-1].endswith(SENT_END): z += 1
            out.append(dict(s=a,e=z,kind=c['kind']+'_split')); a = z
    return finalize(book_id, words, pages, bounds, out, 'structural')

def fixed_chunks(book_id, size=200):
    words, pages = book_stream(book_id)
    bounds = find_boundaries(words)
    out = [dict(s=a, e=min(a+size, len(words)), kind='fixed') for a in range(0, len(words), size)]
    return finalize(book_id, words, pages, bounds, out, 'fixed')

META = {p['book_id']:p for p in passages}
def finalize(book_id, words, pages, bounds, spans, strategy):
    m = META[book_id]; res = []
    for k, c in enumerate(spans):
        if c['e'] <= c['s']: continue
        text = ' '.join(words[c['s']:c['e']])
        res.append(dict(cid=f'{book_id}:{strategy}:{k}', book_id=book_id, subject=m['subject'],
            class_level=m['class_level'], book_title=m['book_title'], chapter=chapter_at(bounds, c['s']) or 'Unlabeled',
            page_start=pages[c['s']], page_end=pages[c['e']-1], kind=c['kind'], strategy=strategy,
            ws=c['s'], we=c['e'], text=text))
    return res

def example_spans(book_id, max_ex=520):
    words, _ = book_stream(book_id)
    b = find_boundaries(words); idx = [j for j,_,_ in b] + [len(words)]
    spans = []
    for t,(j,k,_) in enumerate(b):
        if k == 'example':
            nxt = next(x for x in idx if x > j)
            if nxt-j <= max_ex: spans.append((j, nxt))
    return spans

def split_report(chunks_by_book):
    rows = []
    for bid, ch in chunks_by_book.items():
        ex = example_spans(bid)
        if not ex: continue
        split = sum(1 for s,e in ex if not any(c['ws']<=s and c['we']>=e for c in ch))
        rows.append(dict(book=bid, examples=len(ex), split=split, split_pct=round(100*split/len(ex),1)))
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=['book','examples','split','split_pct'])

ALL_BOOKS = sorted(books)
struct = {b: structural_chunks(b) for b in ALL_BOOKS}
fixed  = {b: fixed_chunks(b)      for b in ALL_BOOKS}
r_s, r_f = split_report(struct), split_report(fixed)

if not r_f.empty and not r_s.empty:
    cmp = r_f.merge(r_s, on=['book','examples'], suffixes=('_fixed','_structural')); display(cmp)
    print('TOTAL examples:', cmp.examples.sum(), '| split (fixed):', cmp.split_fixed.sum(), '| split (structural):', cmp.split_structural.sum())
else:
    print("No 'Example' boundaries detected in the processed textbook pages. Skipping example split merge report.")

print('chunks — fixed:', sum(map(len,fixed.values())), '| structural:', sum(map(len,struct.values())))

def solution_split_report(chunks_by_book, look=150):
    # independent check: is an 'Example' and a 'Solution' appearing after it (<=look words) in different chunks?
    rows=[]
    for bid, ch in chunks_by_book.items():
        words,_ = book_stream(bid); n=tot=0
        owner = np.zeros(len(words), dtype=np.int32)
        for k,c in enumerate(ch): owner[c['ws']:c['we']] = k
        for j,w in enumerate(words):
            if w.strip('.:,')=='Example' and (j==0 or words[j-1].endswith(SENT_END)):
                for t in range(j+1, min(j+look, len(words))):
                    if words[t].strip('.:,')=='Solution':
                        tot+=1; n+= owner[j]!=owner[t]; break
                    if words[t].strip('.:,')=='Example': break
        if tot: rows.append(dict(book=bid, example_with_solution=tot, split=int(n), split_pct=round(100*n/tot,1)))
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=['book','example_with_solution','split','split_pct'])

sf, ss = solution_split_report(fixed), solution_split_report(struct)
if not sf.empty and not ss.empty:
    cmp2 = sf.merge(ss, on=['book','example_with_solution'], suffixes=('_fixed','_structural')); display(cmp2)
    print('Example→Solution TOTAL:', cmp2.example_with_solution.sum(), '| split fixed:', cmp2.split_fixed.sum(), '| split structural:', cmp2.split_structural.sum())
else:
    print("No 'Example → Solution' sequences detected in the processed pages.")

print('NOTE: the first table above and this check are different; in the first, the example boundary is set by the same detector (circular), so the real residual failure shows up here and in the manual audit.')
rnd = random.Random(3)
audit_pool = [c for b in struct for c in struct[b] if c['kind'].startswith('example') or c['kind'].startswith('text')]
if audit_pool:
    for c in rnd.sample(audit_pool, min(len(audit_pool), 3)):
        print('\[manual audit]', c['cid'], 'pp', c['page_start'], '-', c['page_end'], '\...', ' '.join(c['text'].split()[-45:]))

<>:150: SyntaxWarning: invalid escape sequence '\['
<>:150: SyntaxWarning: invalid escape sequence '\.'
<>:150: SyntaxWarning: invalid escape sequence '\['
<>:150: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_2630/3067471671.py:150: SyntaxWarning: invalid escape sequence '\['
  print('\[manual audit]', c['cid'], 'pp', c['page_start'], '-', c['page_end'], '\...', ' '.join(c['text'].split()[-45:]))
/tmp/ipykernel_2630/3067471671.py:150: SyntaxWarning: invalid escape sequence '\.'
  print('\[manual audit]', c['cid'], 'pp', c['page_start'], '-', c['page_end'], '\...', ' '.join(c['text'].split()[-45:]))


,book,examples,split_fixed,split_pct_fixed,split_structural,split_pct_structural
0,1718_balochistan_board_12th_class_biology_text...,1,1,100.0,0,0.0


TOTAL examples: 1 | split (fixed): 1 | split (structural): 0
chunks — fixed: 164 | structural: 117
No 'Example → Solution' sequences detected in the processed pages.
NOTE: the first table above and this check are different; in the first, the example boundary is set by the same detector (circular), so the real residual failure shows up here and in the manual audit.
\[manual audit] 1717_balochistan_board_11th_class_biology_text_book_pdf_taleem360_com_kkkrf:structural:30 pp 41 - 45 \... structure of mitochondria andits role in cellular respiration. Define cytoskeleton and describe its components in detail. Explain the mechanism of movement of cilia and flagella. Illustrate the structure of nucleus. Describe the role of chloroplast in photosynthesis. BIOLOGICAL MOLECULES | 2.1 Biological Molecules in Protoplasm.
\[manual audit] 2861_9th_class_biology_btbb_new_text_book_2026_pdf_taleem360_com_1hggf:structural:1 pp 3 - 3 \... + Mr. Gulzar Ahmed + Mr. Wagar Ahmed + Mr. Wagar Ahmed, + Mrs. Teh

## 3. Building evaluation questions (synthetic proxy) and retrieval metrics
Gold = **the original page from which the question was made**. Having page-level gold allows a fair comparison of both chunking strategies. Metrics: **Recall@k** (the retrieved chunk's page-range covers the gold page) and **MRR**.
This is a *proxy*; the real past-paper set is in Cell 14.

In [ ]:
# ===== Cell 4: Synthetic question generation (LLM) with cache =====
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def usable(p):
    w = p['text'].split()
    return len(w) >= 60 and sum(c.isalpha() for c in p['text'])/len(p['text']) > 0.7

def gen_questions(pool, n, cache_path, lang='en'):
    cache = json.load(open(cache_path)) if os.path.exists(cache_path) else {}
    rnd = random.Random(SEED); pool = [p for p in pool if usable(p)]; rnd.shuffle(pool)
    todo = [p for p in pool[:n] if p['id'] not in cache]
    if todo:
        tok = AutoTokenizer.from_pretrained(QGEN_MODEL, padding_side='left')
        lm = AutoModelForCausalLM.from_pretrained(QGEN_MODEL, torch_dtype=torch.float16, device_map='auto')
        def prompt(t):
            msg = [{'role':'system','content':'You write exam questions for school students.'},
                   {'role':'user','content':'Passage from a school textbook:\n<<<'+t[:1400]+'>>>\n\nWrite ONE clear exam-style question (a student would ask) that this passage answers. '
                    'Do not mention "the passage". Output only the question.'}]
            return tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        B = 16
        for i in tqdm(range(0, len(todo), B), desc='question-gen'):
            batch = todo[i:i+B]
            enc = tok([prompt(p['text']) for p in batch], return_tensors='pt', padding=True).to(lm.device)
            with torch.no_grad(): out = lm.generate(**enc, max_new_tokens=48, do_sample=False)
            for p, o in zip(batch, out):
                q = tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True).strip().split('\n')[0]
                if 8 < len(q) < 300 and q.endswith('?'): cache[p['id']] = q
            json.dump(cache, open(cache_path,'w'))
        del lm; torch.cuda.empty_cache()
    byid = {p['id']:p for p in pool}
    return [dict(qid=k, query=v, book_id=byid[k]['book_id'], subject=byid[k]['subject'], page=byid[k]['page_number'],
                 text=byid[k]['text']) for k,v in cache.items() if k in byid and k in {p['id'] for p in pool[:n]}]

train_pool = [p for p in train_p if p['book_id'] != VAL_BOOK]
val_pool   = [p for p in train_p if p['book_id'] == VAL_BOOK]
Q_train = gen_questions(train_pool, N_TRAIN_QUERIES, f'{OUT}/q_train.json')
Q_val   = gen_questions(val_pool,   N_EVAL_QUERIES,  f'{OUT}/q_val.json')
Q_test  = gen_questions(test_p,     N_EVAL_QUERIES,  f'{OUT}/q_test.json')
print(len(Q_train), len(Q_val), len(Q_test)); print(random.choice(Q_test))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (6 outputs left out)

question-gen:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

question-gen:   0%|          | 0/3 [00:00<?, ?it/s]

39 41 48
{'qid': '2861_9th_class_biology_btbb_new_text_book_2026_pdf_taleem360_com_1hggf_p0009', 'query': 'What are some key areas of focus within the field of biology, and how do they contribute to our understanding of living organisms?', 'book_id': '2861_9th_class_biology_btbb_new_text_book_2026_pdf_taleem360_com_1hggf', 'subject': 'Biology', 'page': 9, 'text': 'Chapter-I: Nature of Science in Biology\n1.4 Branches of Biology\n\nTo study the different aspects of all living organisms, the science of Biology is\nfarther divided into many branches. Some examples are as follows:\n\nMorphology\n\nPharmacology my Anatomy\nGenetics Pryslotoay\nMolecular Erle\nBiology wi [A yology\n\n—— Bicleey) Fee 2xcn0my\n\nMarin Palaeontology\n\nEcology\n\nFig. 1.1: Branches of Biology\n\n1. Cytology (Cell Biology)\n\nCell biology is the study of all the aspects of structure and function of cells. Itis\nalso called cytology.\n2. Embryology\n\nThe study of the development of an embryo to complete individu

In [ ]:
# ===== Cell 5: BM25 floor + metrics =====
from rank_bm25 import BM25Okapi
tokz = lambda t: re.findall(r'\w+', t.lower())

def build_bm25(chunks): return BM25Okapi([tokz(c['text']) for c in chunks])

def hit(chunk, q): return chunk['book_id']==q['book_id'] and chunk['page_start']<=q['page']<=chunk['page_end']

def metrics(rank_fn, queries, ks=(1,3,5,10)):
    # rank_fn(query_dict) -> list of chunk-indices ranked
    rec = {k:[] for k in ks}; rr = []; per = collections.defaultdict(list)
    for q in queries:
        order = rank_fn(q); first = None
        for r, ci in enumerate(order[:max(ks)]):
            if hit(CH[ci], q): first = r+1; break
        for k in ks: rec[k].append(first is not None and first<=k)
        rr.append(1/first if first else 0); per[q['subject']].append(first is not None and first<=5)
    out = {f'R@{k}':round(100*np.mean(v),1) for k,v in rec.items()}; out['MRR']=round(np.mean(rr),3)
    out['per_subject_R@5'] = {s:round(100*np.mean(v),1) for s,v in per.items()}
    return out

def eval_bm25(chunks_by_book, books_subset, queries):
    global CH
    CH = [c for b in books_subset for c in chunks_by_book[b]]
    bm = build_bm25(CH)
    return metrics(lambda q: list(np.argsort(-bm.get_scores(tokz(q['query']))))[:10], queries)

val_books  = [VAL_BOOK]; test_books = sorted(te_books)
print('--- BM25 | test books | FIXED     ->', eval_bm25(fixed,  test_books, Q_test))
print('--- BM25 | test books | STRUCTURAL->', eval_bm25(struct, test_books, Q_test))
print('--- BM25 | val  book  | FIXED     ->', eval_bm25(fixed,  val_books,  Q_val))
print('--- BM25 | val  book  | STRUCTURAL->', eval_bm25(struct, val_books,  Q_val))

--- BM25 | test books | FIXED     -> {'R@1': np.float64(52.1), 'R@3': np.float64(75.0), 'R@5': np.float64(85.4), 'R@10': np.float64(93.8), 'MRR': np.float64(0.662), 'per_subject_R@5': {'Biology': np.float64(85.4)}}
--- BM25 | test books | STRUCTURAL-> {'R@1': np.float64(43.8), 'R@3': np.float64(81.2), 'R@5': np.float64(83.3), 'R@10': np.float64(97.9), 'MRR': np.float64(0.647), 'per_subject_R@5': {'Biology': np.float64(83.3)}}
--- BM25 | val  book  | FIXED     -> {'R@1': np.float64(61.0), 'R@3': np.float64(73.2), 'R@5': np.float64(82.9), 'R@10': np.float64(97.6), 'MRR': np.float64(0.703), 'per_subject_R@5': {'Biology': np.float64(82.9)}}
--- BM25 | val  book  | STRUCTURAL-> {'R@1': np.float64(61.0), 'R@3': np.float64(73.2), 'R@5': np.float64(80.5), 'R@10': np.float64(97.6), 'MRR': np.float64(0.705), 'per_subject_R@5': {'Biology': np.float64(80.5)}}


## 4. Dense retriever — fine-tuning (this is the real "training")
We fine-tune `multilingual-e5-base` on `(question, passage)` pairs with **MultipleNegativesRankingLoss** (in-batch negatives). Each passage is joined with its chapter context (a requirement of the index layer).
Only the train books (excluding VAL_BOOK) are used. The model has never seen the test books.

In [ ]:
# ===== Cell 6: Fine-tune dense retriever =====
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader

def ctx(c):   # embedding text: chapter context + passage
    return f"passage: Class {c['class_level']} {c['subject']} | {c['chapter']} | {c['text']}"
qry = lambda s: 'query: ' + s

FT_DIR = f'{OUT}/e5_ft'
if not os.path.exists(FT_DIR):
    model = SentenceTransformer(DENSE_MODEL, device=DEVICE); model.max_seq_length = 256
    # train positives: page-level passage + its structural chapter context
    ch_lookup = {}
    for b, chs in struct.items():
        for c in chs: ch_lookup.setdefault((b, c['page_start']), c['chapter'])
    ex = []
    for q in Q_train:
        chap = ch_lookup.get((q['book_id'], q['page']), 'Unlabeled')
        m = META[q['book_id']]
        ex.append(InputExample(texts=[qry(q['query']),
              f"passage: Class {m['class_level']} {m['subject']} | {chap} | {q['text']}"]))
    dl = DataLoader(ex, shuffle=True, batch_size=32, drop_last=True)
    loss = losses.MultipleNegativesRankingLoss(model)
    model.fit(train_objectives=[(dl, loss)], epochs=2, warmup_steps=int(0.1*len(dl)*2),
              optimizer_params={'lr': 2e-5}, use_amp=True, show_progress_bar=True)
    model.save(FT_DIR)
ft = SentenceTransformer(FT_DIR, device=DEVICE); base = SentenceTransformer(DENSE_MODEL, device=DEVICE)
ft.max_seq_length = base.max_seq_length = 256
print('fine-tuned model ready')

/tmp/ipykernel_2630/3376789230.py:2: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
/tmp/ipykernel_2630/3376789230.py:2: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

... (11 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

fine-tuned model ready


In [ ]:
# ===== Cell 7: Hybrid retrieval (BM25 + dense, RRF) & comparison table =====
def rrf(rank_lists, k=60, w=None):
    w = w or [1]*len(rank_lists); sc = collections.defaultdict(float)
    for wi, rl in zip(w, rank_lists):
        for r, ci in enumerate(rl): sc[ci] += wi/(k+r+1)
    return sorted(sc, key=sc.get, reverse=True)

class Index:
    def __init__(self, chunks, model):
        self.ch = chunks; self.model = model
        self.bm = build_bm25(chunks)
        self.emb = model.encode([ctx(c) for c in chunks], batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    def dense_scores(self, q):
        qe = self.model.encode([qry(q)], normalize_embeddings=True)[0]; return self.emb @ qe
    def search(self, q, mode='hybrid', k=10):
        d = self.dense_scores(q); bm = self.bm.get_scores(tokz(q))
        dr = list(np.argsort(-d)[:50]); br = list(np.argsort(-bm)[:50])
        order = dr if mode=='dense' else br if mode=='bm25' else rrf([dr, br])
        return order[:k], float(d.max())

def evaluate_modes(chunks_by_book, books_subset, queries, model):
    global CH
    CH = [c for b in books_subset for c in chunks_by_book[b]]
    idx = Index(CH, model); res = {}
    for mode in ('bm25','dense','hybrid'):
        res[mode] = metrics(lambda q, m=mode: idx.search(q['query'], m)[0], queries)
    return res, idx

rows = []
for name, mdl in (('e5 base (no fine-tune)', base), ('e5 FINE-TUNED', ft)):
    for strat, cb in (('fixed', fixed), ('structural', struct)):
        r, _ = evaluate_modes(cb, test_books, Q_test, mdl)
        for mode, m in r.items():
            rows.append(dict(model=name, chunking=strat, mode=mode, **{k:v for k,v in m.items() if k!='per_subject_R@5'}))
results = pd.DataFrame(rows); display(results); results.to_csv(f'{OUT}/results_test_retrieval.csv', index=False)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,model,chunking,mode,R@1,R@3,R@5,R@10,MRR
0,e5 base (no fine-tune),fixed,bm25,52.1,75.0,85.4,93.8,0.662
1,e5 base (no fine-tune),fixed,dense,62.5,79.2,87.5,93.8,0.727
2,e5 base (no fine-tune),fixed,hybrid,62.5,75.0,83.3,95.8,0.723
3,e5 base (no fine-tune),structural,bm25,43.8,81.2,83.3,97.9,0.647
4,e5 base (no fine-tune),structural,dense,50.0,72.9,81.2,93.8,0.638
5,e5 base (no fine-tune),structural,hybrid,45.8,85.4,91.7,100.0,0.655
6,e5 FINE-TUNED,fixed,bm25,52.1,75.0,85.4,93.8,0.662
7,e5 FINE-TUNED,fixed,dense,62.5,79.2,87.5,93.8,0.727
8,e5 FINE-TUNED,fixed,hybrid,62.5,75.0,83.3,95.8,0.723
9,e5 FINE-TUNED,structural,bm25,43.8,81.2,83.3,97.9,0.647


## 5. Reader — restricted to retrieved passages, with citation, abstains on weak retrieval
- The reader answers only from the `[S1]..[S4]` passages; if the answer is not in the passages, it returns `NOT_IN_BOOK`.
- **The citation is built from metadata, not by the model** (book, class, chapter, **page**) — so the LLM cannot invent a wrong page. We only validate the `[Sx]` written by the model against the retrieved set.
- Abstention: dense top-score < threshold (calibrated on validation books + out-of-book questions).

In [ ]:
# ===== Cell 8: Out-of-book questions (refusal evaluation) =====
OOB = [
 "Who won the 1992 Cricket World Cup?","What is the capital of Australia?","How do I make biryani at home?",
 "Write a Python function to reverse a linked list.","Who was the first Prime Minister of Pakistan?",
 "Explain the causes of the French Revolution.","What is the best smartphone to buy this year?",
 "Summarize the plot of Romeo and Juliet.","How many people live in Karachi?","What is the exchange rate of dollar to rupee?",
 "Who painted the Mona Lisa?","Explain the rules of football offside.","What is the stock market?","How do I apply for a passport?",
 "Who wrote the poetry collection Bang-e-Dra?","Describe the history of the Mughal Empire.","What is machine learning?",
 "Translate 'good morning' into French.","What are the symptoms of dengue fever treatment in hospitals?","Who is the current president of the USA?",
 "Explain the Treaty of Versailles.","What is the tallest mountain in Africa?","How do airplanes get booked online?","What is inflation in economics?",
 "Give me a recipe for chocolate cake.","Who invented the telephone company AT&T?","Explain the plot of Hamlet.","What is the GDP of China?",
 "How to start a YouTube channel?","What is the meaning of the Urdu word 'tehzeeb'?","Which team won the FIFA World Cup in 2010?",
 "What are the Fundamental Rights in the Constitution of Pakistan?","Explain Shakespeare's writing style.","How do I learn the guitar?",
 "What is the population of India?","What is blockchain?","Describe the geography of Brazil.","Who discovered America?","What is the Silk Road?",
 "How can I improve my English speaking?"]
json.dump(OOB, open(f'{OUT}/out_of_book_questions.json','w'), indent=1)
print(len(OOB), 'out-of-book questions  (the specification asks for 50: add 10 more of your own)')

40 out-of-book questions  (the specification asks for 50: add 10 more of your own)


In [ ]:
# ===== Cell 9: Calibrate abstention threshold (val book vs OOB) =====
_, val_idx = evaluate_modes(struct, val_books, Q_val[:50], ft)      # build the index
in_scores  = [val_idx.search(q['query'])[1] for q in Q_val]
oob_scores = [val_idx.search(q)[1] for q in OOB]
cands = np.linspace(min(oob_scores+in_scores), max(oob_scores+in_scores), 200)
def f1_at(t):   # balanced score of 'answer in-book' vs 'refuse OOB'
    tpr = np.mean([s>=t for s in in_scores]); tnr = np.mean([s<t for s in oob_scores]); return 2*tpr*tnr/(tpr+tnr+1e-9), tpr, tnr
THRESH = max(cands, key=lambda t: f1_at(t)[0]); f, tpr, tnr = f1_at(THRESH)
print(f'threshold={THRESH:.3f} | answers in-book: {tpr:.1%} (over-refusal {1-tpr:.1%}) | refuses out-of-book: {tnr:.1%}')
json.dump(dict(threshold=float(THRESH), in_book_answer_rate=float(tpr), oob_refusal_rate=float(tnr)), open(f'{OUT}/abstention.json','w'))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

threshold=0.782 | answers in-book: 95.1% (over-refusal 4.9%) | refuses out-of-book: 97.5%


In [ ]:
# ===== Cell 10: Cited reader =====
from transformers import BitsAndBytesConfig
rtok = AutoTokenizer.from_pretrained(READER_MODEL)
rlm  = AutoModelForCausalLM.from_pretrained(READER_MODEL, device_map='auto',
        torch_dtype=torch.float16) if DEVICE=='cuda' else None

SYS = ("You are a revision tutor for Balochistan Board textbooks. Answer ONLY using the numbered sources. "
       "If the sources do not contain the answer, reply exactly: NOT_IN_BOOK. "
       "Be concise (max 5 sentences), give formulas/steps if the source has them, and cite sources like [S1]. "
       "Answer in the same language as the question.")

def cite(c): return f"{c['book_title'].split(' (')[0]}, Chapter: {c['chapter']}, p. {c['page_start']}" + (f"–{c['page_end']}" if c['page_end']!=c['page_start'] else '')

def answer(question, idx, k=4, thresh=None):
    thresh = THRESH if thresh is None else thresh
    order, top = idx.search(question, 'hybrid', k=k)
    res = dict(question=question, top_dense=top, sources=[idx.ch[i] for i in order])
    if top < thresh:
        return {**res, 'answer':'This question was not found in these books (weak retrieval). / Not found in the textbooks.', 'citations':[], 'abstained':True}
    ctx_txt = '\n\n'.join(f"[S{n+1}] ({cite(c)})\n{c['text'][:1500]}" for n,c in enumerate(res['sources']))
    msgs = [{'role':'system','content':SYS},{'role':'user','content':f'Sources:\n{ctx_txt}\n\nQuestion: {question}'}]
    enc = rtok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt', return_dict=True).to(rlm.device)
    with torch.no_grad(): out = rlm.generate(**enc, max_new_tokens=300, do_sample=False)
    txt = rtok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    if 'NOT_IN_BOOK' in txt:
        return {**res, 'answer':'This answer was not found in the passages of the selected books.', 'citations':[], 'abstained':True}
    used = sorted({int(n)-1 for n in re.findall(r'\[S(\d)\]', txt) if 1<=int(n)<=len(res['sources'])})
    cites = [cite(res['sources'][i]) for i in (used or [0])]       # citation = metadata, not a page invented by the model
    return {**res, 'answer':txt, 'citations':cites, 'abstained':False}

# for live checking: idx_all over the whole corpus
CH = [c for b in ALL_BOOKS for c in struct[b]]
idx_all = Index(CH, ft)
pickle.dump(dict(chunks=CH, emb=idx_all.emb), open(f'{OUT}/index_structural.pkl','wb'))
r = answer('State Newton\'s second law of motion.', idx_all); print(r['answer'], '\n→', r['citations'])

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

This answer was not found in the passages of the selected books. 
→ []


## 6. End-to-end evaluation — answer-groundedness, citation correctness (separate), refusal

In [ ]:
# ===== Cell 11: End-to-end proxy evaluation on test books (citation scored SEPARATELY) =====
_, test_idx = evaluate_modes(struct, test_books, Q_test[:50], ft)

def run_e2e(n=60):
    rows = []
    for q in random.Random(1).sample(Q_test, min(n, len(Q_test))):
        r = answer(q['query'], test_idx)
        cited_pages = [(c['book_id'], c['page_start'], c['page_end']) for c in r['sources'][:1]] if not r['abstained'] else []
        # citation correct = the finally displayed (first cited) source covers the gold page
        used = [s for s in r['sources']]
        shown = r['sources'][0] if r['sources'] else None
        rows.append(dict(qid=q['qid'], subject=q['subject'], abstained=r['abstained'],
                         citation_correct=bool(shown and hit(shown, q)) if not r['abstained'] else None,
                         gold_in_top4=any(hit(c,q) for c in r['sources']), answer=r['answer'][:300], question=q['query']))
    return pd.DataFrame(rows)

e2e = run_e2e(); e2e.to_csv(f'{OUT}/e2e_proxy_eval.csv', index=False)
ans = e2e[~e2e.abstained]
print('over-refusal on in-book questions :', f'{e2e.abstained.mean():.1%}')
print('gold page in retrieved top-4      :', f'{e2e.gold_in_top4.mean():.1%}')
print('citation correct (top source page):', f'{ans.citation_correct.mean():.1%}')
oob_r = [answer(q, test_idx)['abstained'] for q in OOB]
print('refusal on out-of-book questions  :', f'{np.mean(oob_r):.1%}')
print('→ to evaluate answer *correctness*, have a teacher score the `answer` column of e2e_proxy_eval.csv (Cell 14).')

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

over-refusal on in-book questions : 22.9%
gold page in retrieved top-4      : 87.5%
citation correct (top source page): 48.6%
refusal on out-of-book questions  : 100.0%
→ to evaluate answer *correctness*, have a teacher score the `answer` column of e2e_proxy_eval.csv (Cell 14).


In [ ]:
# ===== Cell 12: Cross-lingual gap (Urdu queries over the same index) =====
# Machine-translated Urdu proxy. For real Balochi/Urdu queries a native-speaker gold set is needed (spec: "measure on the full set").
def translate(qs, lang='Urdu', model=QGEN_MODEL, B=16):
    tok = AutoTokenizer.from_pretrained(model, padding_side='left')
    lm = AutoModelForCausalLM.from_pretrained(model, torch_dtype=torch.float16, device_map='auto'); out_all=[]
    for i in tqdm(range(0, len(qs), B), desc=f'translate→{lang}'):
        pr = [tok.apply_chat_template([{'role':'user','content':f'Translate to {lang}. Output only the translation:\n{q}'}],
              tokenize=False, add_generation_prompt=True) for q in qs[i:i+B]]
        enc = tok(pr, return_tensors='pt', padding=True).to(lm.device)
        with torch.no_grad(): o = lm.generate(**enc, max_new_tokens=80, do_sample=False)
        out_all += [tok.decode(x[enc.input_ids.shape[1]:], skip_special_tokens=True).strip() for x in o]
    del lm; torch.cuda.empty_cache(); return out_all

sub = Q_test[:200]
ur = translate([q['query'] for q in sub], 'Urdu')
Q_ur = [{**q, 'query': u} for q,u in zip(sub, ur)]
CH = test_idx.ch
gap = pd.DataFrame({'English': metrics(lambda q: test_idx.search(q['query'])[0], sub),
                    'Urdu':    metrics(lambda q: test_idx.search(q['query'])[0], Q_ur)}).drop('per_subject_R@5')
display(gap); gap.to_csv(f'{OUT}/crosslingual_gap.csv')

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

translate→Urdu:   0%|          | 0/3 [00:00<?, ?it/s]

,English,Urdu
R@1,45.8,8.3
R@3,85.4,16.7
R@5,91.7,27.1
R@10,100.0,47.9
MRR,0.655,0.171


## 7. Teacher-gold evaluation (the real evaluation)
Specification: 200 past-paper questions, with the gold answer written by a teacher after finding it in the book, including the page. The cell below creates a template and scores the filled-in CSV for **answer correctness (teacher, 0/1)** and **citation correctness (page-match)** separately.

In [ ]:
# ===== Cell 13/14: Teacher gold template + scoring =====
tpl = f'{GOLD}/gold_eval_TEMPLATE.csv'
if not os.path.exists(tpl):
    pd.DataFrame(columns=['question','language','subject','book_id','edition_year','gold_page','gold_answer',
                          'model_answer','teacher_answer_correct_0_1']).to_csv(tpl, index=False)
    print('Template created:', tpl, '→ fill it in and save it as gold_eval.csv')

gp = f'{GOLD}/gold_eval.csv'
if os.path.exists(gp):
    g = pd.read_csv(gp); rows = []
    for _, r in g.iterrows():
        res = answer(r.question, idx_all)
        pages = [(c['book_id'], c['page_start'], c['page_end']) for c in res['sources'][:1]]
        cite_ok = (not res['abstained']) and any(b==r.book_id and s<=r.gold_page<=e for b,s,e in pages)
        rows.append(dict(**r.to_dict(), abstained=res['abstained'], model_answer=res['answer'], citation_correct=cite_ok))
    G = pd.DataFrame(rows); G.to_csv(f'{OUT}/gold_scored.csv', index=False)
    print('Citation correctness :', G[~G.abstained].citation_correct.mean())
    if 'teacher_answer_correct_0_1' in G and G.teacher_answer_correct_0_1.notna().any():
        print('Answer correctness   :', G.teacher_answer_correct_0_1.mean())
        print(G.groupby('subject')[['citation_correct']].mean())
else:
    print('gold_eval.csv does not exist yet — have a teacher fill it in (week-8 gate G1).')

Template created: /content/ED05_Tutor/gold/gold_eval_TEMPLATE.csv → fill it in and save it as gold_eval.csv
gold_eval.csv does not exist yet — have a teacher fill it in (week-8 gate G1).


## 8. Interface — source page beside the answer (verification = at a glance)

In [2]:
# ===== Cell 15: Gradio interface =====
import os
import gradio as gr
try:
    import fitz
except ModuleNotFoundError:
    raise ModuleNotFoundError("The 'fitz' module (PyMuPDF) is missing. Please run Cell 1 or Cell 1b to install dependencies with '!pip install pymupdf' first.")
import glob

def page_image(c):
    f = os.path.basename(c['source_file']) if 'source_file' in c else None
    pdfp = glob.glob(f"{PDFS}/*{c['book_id']}*") + glob.glob(f"{PDFS}/{META[c['book_id']]['source_file'].split('/')[-1]}")
    if not pdfp: return None
    doc = fitz.open(pdfp[0]); pix = doc[c['page_start']-1].get_pixmap(dpi=110)
    path = f'/tmp/{c["book_id"]}_{c["page_start"]}.png'; pix.save(path); return path

def ui(question):
    r = answer(question, idx_all)
    if r['abstained']: return r['answer'], None, 'no source'
    top = r['sources'][0]
    src = '\n'.join('• '+c for c in r['citations'])
    return r['answer'], page_image(top), f"📖 Source:\n{src}\n(Edition/Year: 2026 listing — verify)"

with gr.Blocks(title='Board Textbook Tutor') as demo:
    gr.Markdown('## 📚 Balochistan Board Revision Tutor\n*This is for revision — not a substitute for class. Always check against the page shown.*')
    q = gr.Textbox(label='Question (Urdu / English)'); b = gr.Button('Ask')
    with gr.Row():
        a = gr.Textbox(label='Answer', lines=8); img = gr.Image(label='Source page')
    s = gr.Textbox(label='Citation'); b.click(ui, q, [a, img, s])
demo.launch(share=True)

ModuleNotFoundError: The 'fitz' module (PyMuPDF) is missing. Please run Cell 1 or Cell 1b to install dependencies with '!pip install pymupdf' first.

## 9. Release checklist (per the specification)
- ✅ Release: pipeline code, index-building code, evaluation set, results — **not the books' text** (`index_structural.pkl` contains text → **do not make it public**).
- ✅ Page provenance assertion, extraction audit, split-example report, citation scored separately.
- ⚠️ Synthetic-proxy numbers ≠ teacher-gold numbers. State both clearly and separately in the report.

## 10. Download your results to your PC
Because uploaded files live on Colab's temporary disk, run this cell before closing the session to save everything (results, caches, scored CSVs, fine-tuned model) to your computer as a zip. Skip it if you used `DATA_SOURCE = 'drive'`.

In [ ]:
# ===== Cell 16: Download outputs to your PC =====
if IN_COLAB and DATA_SOURCE == 'upload':
    shutil.make_archive('/content/ED05_outputs', 'zip', OUT)
    files.download('/content/ED05_outputs.zip')
else:
    print('Nothing to download: outputs are already in', OUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>